# RoadVision AI — vehicle detection

Run sections in order. This notebook covers dataset inspection, preparation, and optional YOLOv9 training. The source dataset is never edited.

## 1. Project setup
Local: launch Jupyter from the project root. Colab: this cell clones the project when needed.

In [ ]:
from pathlib import Path
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content/RoadVision-AI-")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/Oussamatiu/RoadVision-AI-.git", str(ROOT)], check=True)
else:
    current = Path.cwd().resolve()
    ROOT = next((p for p in [current, *current.parents] if (p / "src/dataset.py").is_file()), None)
    if ROOT is None:
        raise RuntimeError("Launch Jupyter from the RoadVision-AI- project folder.")
print("Project:", ROOT)

In [ ]:
# Install into the current notebook kernel. Restart the kernel if prompted.
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")], check=True)

In [ ]:
import os
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
import kagglehub

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.dataset import load_classes, inspect_dataset, prepare_dataset, class_counts

SEED = 42
TEST_FRACTION = 0.15
DROP_EMPTY = False
SPLIT_ROOT = ROOT / "data" / "road_vehicle_split"
for folder in ["data", "models", "reports", "third_party"]:
    (ROOT / folder).mkdir(exist_ok=True)

## 2. Download and explore
The download stays in the Kaggle cache. If authentication is requested, configure Kaggle credentials in your runtime.

In [ ]:
dataset = Path(kagglehub.dataset_download("ashfakyeafi/road-vehicle-images-dataset"))
print("Dataset:", dataset)
for item in sorted(dataset.iterdir()):
    print(item.name)

configs = sorted(set(dataset.rglob("*.yaml")) | set(dataset.rglob("*.yml")))
print("Configuration files:")
for path in configs:
    print(path.relative_to(dataset))
# Set YAML_PATH explicitly if the download contains several dataset configurations.
preferred = [p for p in configs if p.name == "data_1.yaml"]
choices = preferred or [p for p in configs if p.name in {"data.yaml", "data.yml"}]
if len(choices) != 1:
    raise ValueError("Choose the dataset YAML from the printed paths and set YAML_PATH manually.")
YAML_PATH = choices[0]
class_names = load_classes(YAML_PATH)
display(class_names)

## 3. Validate images and annotations
Check matching labels, class IDs, normalized boxes, and image decoding. Invalid pairs are reported and excluded from the prepared copy. Empty labels are valid background examples.

In [ ]:
records = inspect_dataset(dataset, class_names)
if not records:
    raise ValueError("No images found in the dataset.")
issues = pd.DataFrame([
    {"image": str(r["image"].relative_to(dataset)), "errors": "; ".join(r["errors"])}
    for r in records if r["errors"]
], columns=["image", "errors"])
print("Images:", len(records))
print("Invalid pairs:", len(issues))
print("Empty labels:", sum(r["empty"] for r in records))
display(issues.head(20))
issues.to_csv(ROOT / "reports" / "dataset_issues.csv", index=False)

usable = [r for r in records if not r["errors"] and not (DROP_EMPTY and r["empty"])]
if not usable:
    raise ValueError("No usable image-label pairs remain.")
with Image.open(usable[0]["image"]) as image:
    plt.figure(figsize=(9, 6))
    plt.imshow(image.convert("RGB"))
plt.axis("off")
plt.title("Example vehicle image")
plt.show()

## 4. Class distribution and dataset splits
Keep the existing validation and test sets. If no test set exists, reserve 15% of training candidates. Choose a new `SPLIT_ROOT` to rerun preparation; existing folders are never overwritten.

In [ ]:
counts = class_counts(usable)
distribution = pd.DataFrame([
    {"Class": name, "Objects": counts[class_id]}
    for class_id, name in class_names.items()
])
display(distribution)
distribution.plot.bar(x="Class", y="Objects", legend=False, figsize=(10, 5))
plt.title("RoadVision AI — class distribution before splitting")
plt.ylabel("Objects")
plt.tight_layout()
plt.show()

In [ ]:
DATA_YAML, split_counts = prepare_dataset(
    records, SPLIT_ROOT, class_names,
    test_fraction=TEST_FRACTION, seed=SEED, drop_empty=DROP_EMPTY,
)
for split, count in split_counts.items():
    images = list((SPLIT_ROOT / "images" / split).iterdir())
    labels = list((SPLIT_ROOT / "labels" / split).glob("*.txt"))
    assert len(images) == len(labels) == count
    assert {p.stem for p in images} == {p.stem for p in labels}
print(split_counts)
print(DATA_YAML.read_text())

## 5. Optional YOLOv9 setup
Set `RUN_TRAINING = True` to download YOLOv9, install its dependencies, and train. This can take substantial time and requires a CUDA GPU. Upstream YOLOv9 dependencies may need adjustment for your Python/PyTorch runtime. Only load model weights from a trusted source.

In [ ]:
RUN_TRAINING = False
YOLO_ROOT = ROOT / "third_party" / "yolov9"
WEIGHTS = ROOT / "models" / "yolov9-c.pt"
EPOCHS = 10
BATCH_SIZE = 4
IMAGE_SIZE = 640

if RUN_TRAINING:
    if not YOLO_ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/WongKinYiu/yolov9.git", str(YOLO_ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(YOLO_ROOT / "requirements.txt")], check=True)
    import torch
    print("PyTorch:", torch.__version__)
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a CUDA GPU runtime before training.")
    print("GPU:", torch.cuda.get_device_name(0))
    if not WEIGHTS.is_file():
        from urllib.request import urlretrieve
        temporary = WEIGHTS.with_suffix(".download")
        urlretrieve("https://github.com/WongKinYiu/yolov9/releases/download/v0.1/yolov9-c.pt", temporary)
        temporary.replace(WEIGHTS)
else:
    print("Training disabled. Dataset preparation is complete.")

## 6. Train
Each run uses a unique output name. Best and last model checkpoints are stored under its `weights/` directory.

In [ ]:
from datetime import datetime, timezone

if RUN_TRAINING:
    RUN_NAME = "roadvision_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
    RESULT_DIR = ROOT / "reports" / RUN_NAME
    env = dict(os.environ, WANDB_MODE="disabled", WANDB_DISABLED="true")
    subprocess.run([
        sys.executable, "train_dual.py",
        "--weights", str(WEIGHTS),
        "--cfg", "models/detect/yolov9-c.yaml",
        "--data", str(DATA_YAML),
        "--hyp", "data/hyps/hyp.scratch-high.yaml",
        "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
        "--img", str(IMAGE_SIZE), "--device", "0", "--workers", "2",
        "--project", str(ROOT / "reports"), "--name", RUN_NAME,
    ], cwd=YOLO_ROOT, env=env, check=True)

## 7. View training results
These are training/validation curves, not a held-out test evaluation. The test split remains available in `data.yaml` for later evaluation.

In [ ]:
if RUN_TRAINING:
    csv_path = RESULT_DIR / "results.csv"
    if csv_path.is_file():
        results = pd.read_csv(csv_path)
        results.columns = results.columns.str.strip()
        display(results.tail())
    plot_path = RESULT_DIR / "results.png"
    if plot_path.is_file():
        with Image.open(plot_path) as image:
            plt.figure(figsize=(16, 10))
            plt.imshow(image)
        plt.axis("off")
        plt.show()
    else:
        print("No results.png found; inspect the run CSV and training output.")
    print("Run directory:", RESULT_DIR)